# Chapter 3: Build the smallest useful runtime

Companion to *Agent Harness Engineering*. Listings 3.1 to 3.8 live in `ahe/ch03.py`.

## Setup

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd()
if not (root / "ahe").exists():
    root = root.parent            # opened from inside chapter-0N/
sys.path.insert(0, str(root))
assert sys.version_info >= (3, 10), "Use Python 3.10 or newer."
import os
LIVE = os.environ.get("AHE_LIVE") == "1"
print("Python", sys.version.split()[0], "| live calls:", "on" if LIVE else "off")

## 1. Fixed program versus planner (Table 3.1)

In [ ]:
from ahe import ch03
from ahe.ch03 import request, run_task, fixed_screen, fake_planner, packet
for pid in ("P041", "P043"):
    fixed = fixed_screen(request(pid, 3))
    planned = run_task(request(pid, 3), fake_planner)
    print(pid, "fixed:", fixed.results["no_anticoag"].status, "| planner:",
          planned.results["no_anticoag"].status, "reads", planned.reads)

## 2. Misbehaving planners (Table 3.3)

In [ ]:
for row in ch03.trials(request("P041", 3)):
    print(f"{row[0]:14s} {row[1]:10s} {row[2][:40]:40s} steps={row[3]:2d} reads={row[4]}")

## 3. One run's raw events

In [ ]:
run = run_task(request("P041", 3), fake_planner)
for e in run.events:
    print(e)
print(packet(run))

## 4. Replay the recorded live-planner runs (Table 3.5)

The runtime is deterministic given the planner's replies, so feeding the recorded replies back recomputes each run. Every call must rebuild exactly the prompt that was recorded, and every reply must be used; otherwise the cell stops. The saved Chapter 3 records hold calls, not outcomes, so this is a recomputation, checked call by call.

In [ ]:
import json
from ahe import live
trials = ["P041"] * 5 + ["P043"] * 5
runs, _ = live.replay("ch03_planner_gpt-6-luna", trials,
                      lambda c: ch03.PLANNER_PROMPT + json.dumps(c, default=str),
                      lambda planner, pid: run_task(request(pid, 3), planner))
for pid, r in zip(trials, runs):
    print(pid, r.state, r.count("proposal"), "steps,", r.reads, "reads,",
          r.results.get("no_anticoag", ("-",))[0])

### Live calls

Every live cell below does nothing unless `AHE_LIVE=1` is set before Jupyter starts. Settings come from the environment or from `~/.config/ahe/openrouter.env`:

```
AHE_BASE_URL=https://openrouter.ai/api/v1     # or https://<workspace>/serving-endpoints
AHE_API_KEY=...                                # never commit this file
AHE_MODEL=openai/gpt-6-luna
```

Any OpenAI-compatible endpoint works, including a Databricks workspace (base URL `https://<workspace-host>/serving-endpoints`, a personal access token as the key, and a served model name). `ahe.live.Recorder` replaces `chat` so that each call is appended to `runs/<experiment>.jsonl` with its prompt, reply, token counts, and cost. Use a new experiment name for your own runs so the book's records stay as they are.

In [ ]:
if LIVE:
    live.load_env()
    ch01.chat = live.Recorder("my_ch03_planner")
    r = run_task(request("P043", 3), ch03.live_planner)
    for e in r.events:
        if e["kind"] == "proposal":
            print(e["proposal"])
    print(r.state, r.reason)
else:
    print("Set AHE_LIVE=1 to run the live planner.")

## Exercise solutions

### Exercise 3.1

A rule: when the step budget is spent and `unfinished(run)` is empty, end the run as completed with the reason "budget spent; every rule current". The decision has to be made before the stop is recorded, so the loop below is Chapter 3's `drive` with that one branch changed. It is wrong when finishing is itself a decision the planner owes, for example when a later chapter requires a drafted packet or an approval before completion.

In [ ]:
def drive_autofinish(run, planner, reader=ch03.fake_reader):
    while run.state == "running":
        if run.count("proposal") >= run.budget.steps:
            if ch03.unfinished(run):
                ch03.end(run, "blocked", "step budget spent")
            else:
                ch03.end(run, "completed", "budget spent; every rule current")
        else:
            try:
                ch03.step(run, planner, reader)
            except ch03.BudgetExhausted as why:
                run.record("observation", error=str(why))
                ch03.end(run, "blocked", str(why))
    return run

def never_finishes(context):       # does the work, never asks to finish
    p = ch03.fake_planner(context)
    return p if p["tool"] != "finish" else {"tool": "search_notes", "args": {"terms": ["zzzz"]}}
r = drive_autofinish(ch03.start(request("P041", 3), ch03.Budget(steps=9)), never_finishes)
print(r.state, r.reason, "| stops:", r.count("stop"))

### Exercise 3.2

The gate needs `draft_packet` in the run's registry with no arguments, and the completion check must now also require a draft made after the last change to the evidence. The planner below drafts before it asks to finish.

In [ ]:
def t_draft(run):
    run.draft, run.draft_version = packet(run), run.version
    return {"drafted": sorted(run.draft["criteria"])}
registry = ch03.Registry({**ch03.TOOLS, "draft_packet": t_draft},
                         {**ch03.SPEC, "draft_packet": {}})

def drafting_planner(context):
    p = ch03.fake_planner(context)
    drafted = any("drafted" in e for e in context["observations"])
    if p["tool"] == "finish" and not drafted:
        return {"tool": "draft_packet", "args": {}}
    return p

def step_needing_a_draft(run, planner, reader=ch03.fake_reader):
    if planner(ch03.context_for(run))["tool"] == "finish" and \
            getattr(run, "draft_version", None) != run.version:
        run.record("proposal", proposal={"tool": "finish"})
        run.record("observation", error="cannot finish: no current draft")
        return
    ch03.step(run, planner, reader)

run = ch03.start(request("P041", 3), registry=registry)
while run.state == "running" and run.count("proposal") < 14:
    step_needing_a_draft(run, drafting_planner)
print(run.state, [e.get("tool") for e in run.events if e["kind"] == "observation"][-3:])

### Exercise 3.3

Read the proposals the replay prints in section 4. The gate bounds the search terms and `ground` checks what the reads find, but neither measures recall: a planner that searches for the wrong words, reads nothing, and evaluates gets `unknown` without any check failing. Before letting a model choose search terms unwatched, write fixtures that name the notes each search must find, including a brand name and a drug outside the term list. The harness already refuses to conclude absence from a search alone: `met` for the note rule needs every note in scope read.